In [4]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score


In [5]:
# 1. Loading the individual tables
results = pd.read_csv(r'datasets\results.csv')
races = pd.read_csv(r'datasets\races.csv')
drivers = pd.read_csv(r'datasets\drivers.csv')
constructors = pd.read_csv(r'datasets\constructors.csv')
circuits = pd.read_csv(r'datasets\circuits.csv')
status = pd.read_csv(r'datasets\status.csv')


In [6]:
results.head()
races.head()
drivers.head()  
# constructors.head()

,driverId,driverRef,number,code,forename,surname,dob,nationality,url
0,1,hamilton,44,HAM,Lewis,Hamilton,1985-01-07,British,http://en.wikipedia.org/wiki/Lewis_Hamilton
1,2,heidfeld,\N,HEI,Nick,Heidfeld,1977-05-10,German,http://en.wikipedia.org/wiki/Nick_Heidfeld
2,3,rosberg,6,ROS,Nico,Rosberg,1985-06-27,German,http://en.wikipedia.org/wiki/Nico_Rosberg
3,4,alonso,14,ALO,Fernando,Alonso,1981-07-29,Spanish,http://en.wikipedia.org/wiki/Fernando_Alonso
4,5,kovalainen,\N,KOV,Heikki,Kovalainen,1981-10-19,Finnish,http://en.wikipedia.org/wiki/Heikki_Kovalainen


In [7]:
# 2. Merging Results with Races
# Pull year, round, circuitId and date for temporal and feature engineering work.
df = pd.merge(
    results,
    races[['raceId', 'year', 'round', 'circuitId', 'date']],
    on='raceId',
    how='left'
)
df.head()


,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,...,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId,year,round,circuitId,date
0,1,18,1,1,22,1,1,1,1,10.0,...,5690616,39,2,1:27.452,218.300,1,2008,1,1,2008-03-16
1,2,18,2,2,3,5,2,2,2,8.0,...,5696094,41,3,1:27.739,217.586,1,2008,1,1,2008-03-16
2,3,18,3,3,7,7,3,3,3,6.0,...,5698779,41,5,1:28.090,216.719,1,2008,1,1,2008-03-16
3,4,18,4,4,5,11,4,4,4,5.0,...,5707797,58,7,1:28.603,215.464,1,2008,1,1,2008-03-16
4,5,18,5,1,23,3,5,5,5,4.0,...,5708630,43,1,1:27.418,218.385,1,2008,1,1,2008-03-16


In [8]:
# 3. Merge with Drivers
# Pulling in the driver's reference name so we know who is who
df = pd.merge(df, drivers[['driverId', 'driverRef']], on='driverId', how='left')
df.head()


,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,...,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId,year,round,circuitId,date,driverRef
0,1,18,1,1,22,1,1,1,1,10.0,...,39,2,1:27.452,218.300,1,2008,1,1,2008-03-16,hamilton
1,2,18,2,2,3,5,2,2,2,8.0,...,41,3,1:27.739,217.586,1,2008,1,1,2008-03-16,heidfeld
2,3,18,3,3,7,7,3,3,3,6.0,...,41,5,1:28.090,216.719,1,2008,1,1,2008-03-16,rosberg
3,4,18,4,4,5,11,4,4,4,5.0,...,58,7,1:28.603,215.464,1,2008,1,1,2008-03-16,alonso
4,5,18,5,1,23,3,5,5,5,4.0,...,43,1,1:27.418,218.385,1,2008,1,1,2008-03-16,kovalainen


In [9]:
# 4. Merge with Constructors and Circuits
# Pulling in team and circuit geo fields for additional feature engineering.
df = pd.merge(df, constructors[['constructorId', 'name']], on='constructorId', how='left')
df.rename(columns={'name': 'constructor_name'}, inplace=True)

df = pd.merge(df, circuits[['circuitId', 'name', 'country', 'lat']], on='circuitId', how='left')
df.rename(columns={'name': 'circuit_name'}, inplace=True)

print('Data merged successfully!')
print(f'Total records: {df.shape[0]}')
display(df.head())



Data merged successfully!
Total records: 26759


,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,...,statusId,year,round,circuitId,date,driverRef,constructor_name,circuit_name,country,lat
0,1,18,1,1,22,1,1,1,1,10.0,...,1,2008,1,1,2008-03-16,hamilton,McLaren,Albert Park Grand Prix Circuit,Australia,-37.8497
1,2,18,2,2,3,5,2,2,2,8.0,...,1,2008,1,1,2008-03-16,heidfeld,BMW Sauber,Albert Park Grand Prix Circuit,Australia,-37.8497
2,3,18,3,3,7,7,3,3,3,6.0,...,1,2008,1,1,2008-03-16,rosberg,Williams,Albert Park Grand Prix Circuit,Australia,-37.8497
3,4,18,4,4,5,11,4,4,4,5.0,...,1,2008,1,1,2008-03-16,alonso,Renault,Albert Park Grand Prix Circuit,Australia,-37.8497
4,5,18,5,1,23,3,5,5,5,4.0,...,1,2008,1,1,2008-03-16,kovalainen,McLaren,Albert Park Grand Prix Circuit,Australia,-37.8497


In [10]:
# 2) Show as a vertical list (easier to read)
for c in df.columns:
    print(c)


resultId
raceId
driverId
constructorId
number
grid
position
positionText
positionOrder
points
laps
time
milliseconds
fastestLap
rank
fastestLapTime
fastestLapSpeed
statusId
year
round
circuitId
date
driverRef
constructor_name
circuit_name
country
lat


In [11]:
# 5. Data Cleaning + Expanded Feature Engineering

# Keep modern F1 era for better relevance.
df = df[df['year'] >= 2010].copy()

# Core numeric conversions.
for col in ['grid', 'positionOrder', 'statusId', 'year', 'round', 'lat']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
# changing date to datetime and dropping rows with critical missing values in key columns.
df['date'] = pd.to_datetime(df['date'], errors='coerce') 
df = df.dropna(subset=['grid', 'positionOrder', 'circuitId', 'constructorId', 'driverId', 'year', 'round'])




In [12]:
# Target variable.
df['Podium'] = (df['positionOrder'] <= 3).astype(int)

# 1) Weather forecast code: Dry=0, Mixed=1, Wet=2 (calendar + hemisphere proxy)
df['month'] = df['date'].dt.month.fillna(6).astype(int)
wet_months = [6, 7, 8, 9]
mixed_months = [3, 4, 5, 10, 11]

df['weather_code'] = np.select(
    [df['month'].isin(wet_months), df['month'].isin(mixed_months)],
    [2, 1],
    default=0,
).astype(int)

# Southern hemisphere season inverted: wet months are Dec-Feb, mixed are Mar-May & Sep-Nov.
df.loc[(df['lat'] < 0) & (df['month'].isin([12, 1, 2])), 'weather_code'] = 2

# 2) Tyre strategy code: Conservative=0, Balanced=1, Aggressive=2
# Front rows often run balanced setups, midfield/back rows need aggressive offsets.
df['tyre_strategy_code'] = np.select(
    [df['grid'] <= 4, (df['grid'] >= 5) & (df['grid'] <= 10)],
    [1, 2],
    default=0,
).astype(int)

# 3) Recent form (0-100): rolling podium rate over last 5 races per driver.
df = df.sort_values(['driverId', 'year', 'round']).reset_index(drop=True)
df['recent_form'] = (
    df.groupby('driverId')['Podium']
    .transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
    .fillna(0.30)
    * 100
)

# 4) Reliability risk (0-100): rolling non-finish proxy by constructor.
# statusId == 1 means finished in Ergast status table.
df['dnf_flag'] = (df['statusId'] != 1).astype(int)
df['reliability_risk'] = (
    df.groupby('constructorId')['dnf_flag']
    .transform(lambda s: s.shift(1).rolling(10, min_periods=1).mean())
    .fillna(0.15)
    * 100
)

# 5) Pit crew rating (1-10): inverse of recent reliability risk with light smoothing.
df['pit_crew_rating'] = (10 - (df['reliability_risk'] / 10)).clip(1, 10)

# 6) Aggression level (0-100): rolling places gained trend.
df['places_gained'] = (df['grid'] - df['positionOrder']).clip(-10, 10)
df['aggression_level'] = (
    df.groupby('driverId')['places_gained']
    .transform(lambda s: s.shift(1).rolling(8, min_periods=1).mean())
    .fillna(0)
)
df['aggression_level'] = ((df['aggression_level'] + 10) * 5).clip(0, 100)

# 7) Teammate pressure (0-100): when teammate's recent form is stronger.
df['teammate_best_form'] = df.groupby(['constructorId', 'year', 'round'])['recent_form'].transform('max')
df['teammate_pressure'] = (50 + (df['teammate_best_form'] - df['recent_form'])).clip(0, 100)

# Final training matrix
feature_cols = [
    'grid',
    'circuitId',
    'constructorId',
    'driverId',
    'weather_code',
    'tyre_strategy_code',
    'pit_crew_rating',
    'recent_form',
    'reliability_risk',
    'aggression_level',
    'teammate_pressure',
]

df_model = df[feature_cols + ['Podium']].copy()

for col in feature_cols + ['Podium']:
    df_model[col] = pd.to_numeric(df_model[col], errors='coerce')

df_model = df_model.dropna().reset_index(drop=True)

print('Feature Engineering Complete!')
print(f'Total rows ready for training: {df_model.shape[0]}')
print('\nClass Balance (0 = No Podium, 1 = Podium):')
print(df_model['Podium'].value_counts(normalize=True).rename('ratio'))
display(df_model.head())

Feature Engineering Complete!
Total rows ready for training: 6436

Class Balance (0 = No Podium, 1 = Podium):
Podium
0    0.857831
1    0.142169
Name: ratio, dtype: float64


,grid,circuitId,constructorId,driverId,weather_code,tyre_strategy_code,pit_crew_rating,recent_form,reliability_risk,aggression_level,teammate_pressure,Podium
0,4,3,1,1,1,1,8.5,30.000000,15.0,50.000000,50.0,1
1,11,1,1,1,1,0,10.0,100.000000,0.0,55.000000,50.0,0
2,20,2,1,1,1,0,10.0,50.000000,0.0,65.000000,50.0,0
3,6,17,1,1,1,2,10.0,33.333333,0.0,76.666667,50.0,1
4,3,4,1,1,1,1,10.0,50.000000,0.0,75.000000,50.0,0


In [13]:
# 6. Train / Evaluate / Export
X = df_model[feature_cols]
y = df_model['Podium']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=18,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

pred = rf.predict(X_test)
proba = rf.predict_proba(X_test)[:, 1]

print(f'Accuracy: {accuracy_score(y_test, pred):.4f}')
print(f'ROC-AUC: {roc_auc_score(y_test, proba):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, pred))

importance_df = (
    pd.DataFrame({'Feature': feature_cols, 'Importance': rf.feature_importances_})
    .sort_values('Importance', ascending=False)
)
print('\nTop feature importances:')
display(importance_df.head(10))

joblib.dump(rf, 'f1_model.pkl')
joblib.dump(feature_cols, 'f1_model_features.pkl')
print("Model written to f1_model.pkl")
print("Feature list written to f1_model_features.pkl")


Accuracy: 0.9053
ROC-AUC: 0.9311

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.96      0.95      1105
           1       0.72      0.55      0.62       183

    accuracy                           0.91      1288
   macro avg       0.82      0.76      0.78      1288
weighted avg       0.90      0.91      0.90      1288


Top feature importances:


,Feature,Importance
0,grid,0.261100
7,recent_form,0.175149
9,aggression_level,0.095300
8,reliability_risk,0.087892
6,pit_crew_rating,0.081618
1,circuitId,0.081339
3,driverId,0.069591
5,tyre_strategy_code,0.058436
2,constructorId,0.043496
10,teammate_pressure,0.027398


Model written to f1_model.pkl
Feature list written to f1_model_features.pkl


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

X = df_model[feature_cols]
y = df_model['Podium']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=18,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)

    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1 Score": f1_score(y_test, pred, zero_division=0),
        "ROC AUC": roc_auc_score(y_test, proba)
    })

comparison_df = pd.DataFrame(results).sort_values("ROC AUC", ascending=False)

comparison_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC
2,Gradient Boosting,0.911491,0.722581,0.612022,0.662722,0.937683
1,Random Forest,0.905280,0.719424,0.546448,0.621118,0.931098
0,Logistic Regression,0.897516,0.686131,0.513661,0.587500,0.925218


In [15]:
comparison_df.to_csv("model_comparison.csv", index=False)